# Análisis Exploratorio — NASA CMAPSS Fleet Dataset

**Dataset**: C-MAPSS (Commercial Modular Aero-Propulsion System Simulation)  
**Fuente**: NASA Ames Prognostics Data Repository  
**Referencia**: Saxena et al., *Damage Propagation Modeling for Aircraft Engine Run-to-Failure Simulation*, PHM08, 2008

---

## Objetivo

Antes de diseñar cualquier modelo predictivo se requiere entender la estructura del dato: qué mide, cómo varía, qué patrones existen y qué decisiones de preprocesamiento son necesarias. Este análisis cubre los cuatro subconjuntos del repositorio (FD001–FD004), que representan distintas combinaciones de condiciones operacionales y modos de falla.

| Subconjunto | Motores (train) | Condiciones operacionales | Modos de falla |
|-------------|-----------------|--------------------------|----------------|
| FD001       | 100             | 1 (nivel del mar)        | HPC Degradation |
| FD002       | 260             | 6                        | HPC Degradation |
| FD003       | 100             | 1 (nivel del mar)        | HPC + Fan Degradation |
| FD004       | 248             | 6                        | HPC + Fan Degradation |

El conjunto de entrenamiento incluye trayectorias completas hasta el fallo. El conjunto de prueba se trunca antes del fallo; el archivo `RUL_FDxxx.txt` provee el RUL verdadero en el último ciclo registrado.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

## Nomenclatura de columnas

Cada fila del archivo corresponde a un ciclo operacional de un motor específico. Los 26 campos son:

| # | Variable | Descripción física |
|---|----------|--------------------|
| 1 | `unit` | ID del motor |
| 2 | `cycle` | Ciclo operacional (tiempo discreto) |
| 3 | `os1` | Altitud de vuelo (kft) |
| 4 | `os2` | Número de Mach |
| 5 | `os3` | Temperatura estranguladora (°R) |
| 6–26 | `s1`–`s21` | 21 mediciones de sensores (temperatura, presión, velocidad, flujo) |

Los sensores monitorean variables físicas en distintos puntos del motor turbofán: temperatura de salida del compresor de alta presión (`s3`), velocidad del núcleo (`s9`), presión estática en el HPC (`s11`), entre otros. No todos los sensores son informativos — varios tienen varianza prácticamente nula a lo largo de toda la operación.

In [ ]:
# Definición de columnas según documentación del dataset
COLS = [
    'unit', 'cycle',
    'os1', 'os2', 'os3',
    's1',  's2',  's3',  's4',  's5',  's6',  's7',
    's8',  's9',  's10', 's11', 's12', 's13', 's14',
    's15', 's16', 's17', 's18', 's19', 's20', 's21'
]

SENSORES = [f's{i}' for i in range(1, 22)]

# Ruta relativa al directorio de datos
DATA_DIR = Path('data/CMaps')

# Descripción física de cada sensor (para etiquetas en gráficas)
DESC_SENSOR = {
    's1':  'T2 — Temp. entrada fan (°R)',
    's2':  'T24 — Temp. salida LPC (°R)',
    's3':  'T30 — Temp. salida HPC (°R)',
    's4':  'T50 — Temp. salida LPT (°R)',
    's5':  'P2 — Presión entrada fan (psia)',
    's6':  'P15 — Presión bypass (psia)',
    's7':  'P30 — Presión salida HPC (psia)',
    's8':  'Nf — Velocidad fan (rpm)',
    's9':  'Nc — Velocidad núcleo (rpm)',
    's10': 'epr — Relación de presión',
    's11': 'Ps30 — Presión estática HPC (psia)',
    's12': 'phi — Flujo comb./Ps30 (pps/psi)',
    's13': 'NRf — Velocidad fan corregida (rpm)',
    's14': 'NRc — Velocidad núcleo corregida (rpm)',
    's15': 'BPR — Bypass ratio',
    's16': 'farB — Relación aire-comb. quemador',
    's17': 'htBleed — Entalpía de sangrado',
    's18': 'Nf_dmd — Velocidad fan demandada',
    's19': 'PCNfR_dmd — Vel. fan corregida demandada',
    's20': 'W31 — Sangrado refrigeración HPT (lbm/s)',
    's21': 'W32 — Sangrado refrigeración LPT (lbm/s)',
}


def cargar_fd(n: int) -> tuple:
    """Carga los archivos train, test y RUL para el subconjunto FDn."""
    leer = lambda nombre: pd.read_csv(
        DATA_DIR / nombre, sep=r'\s+', header=None, names=COLS
    )
    train = leer(f'train_FD{n:03d}.txt')
    test  = leer(f'test_FD{n:03d}.txt')
    rul   = pd.read_csv(
        DATA_DIR / f'RUL_FD{n:03d}.txt', header=None, names=['RUL_final']
    )
    return train, test, rul

In [ ]:
# Cargar los cuatro subconjuntos y agregar variables derivadas
datasets = {}

for n in [1, 2, 3, 4]:
    train, test, rul = cargar_fd(n)

    # RUL en entrenamiento: distancia al último ciclo de cada motor
    max_ciclo_tr = train.groupby('unit')['cycle'].transform('max')
    train['RUL']     = max_ciclo_tr - train['cycle']
    train['distress'] = (train['RUL'] < 20).astype(int)
    train['dataset']  = f'FD{n:03d}'
    train['split']    = 'train'

    # RUL en prueba: RUL_final proviene del archivo externo
    # RUL en cada ciclo = RUL_final + (max_ciclo_test - cycle)
    max_ciclo_te = test.groupby('unit')['cycle'].transform('max')
    rul_map = dict(enumerate(rul['RUL_final'].values, start=1))
    test['RUL']      = test['unit'].map(rul_map) + (max_ciclo_te - test['cycle'])
    test['distress']  = (test['RUL'] < 20).astype(int)
    test['dataset']   = f'FD{n:03d}'
    test['split']     = 'test'

    datasets[f'FD{n:03d}'] = {'train': train, 'test': test, 'rul': rul}

train_all = pd.concat([v['train'] for v in datasets.values()], ignore_index=True)
test_all  = pd.concat([v['test']  for v in datasets.values()], ignore_index=True)
all_data  = pd.concat([train_all, test_all], ignore_index=True)

print(f'Registros totales : {len(all_data):>8,}')
print(f'  Train           : {len(train_all):>8,}')
print(f'  Test            : {len(test_all):>8,}')
print(f'Columnas          : {list(all_data.columns)}')

## 1. Inventario general

El primer paso es cuantificar la flota: cuántos motores hay por subconjunto, cuántos ciclos opera cada uno en promedio y cuál es el rango de vidas. Esta información determina parámetros clave del modelado — tamaño de ventana deslizante, número de muestras disponibles para validación cruzada y grado esperado de desbalance de clases.

In [ ]:
filas = []
for nombre, d in datasets.items():
    tr   = d['train']
    te   = d['test']
    vida = tr.groupby('unit')['cycle'].max()
    filas.append({
        'Dataset'             : nombre,
        'Motores (train)'     : tr['unit'].nunique(),
        'Motores (test)'      : te['unit'].nunique(),
        'Ciclos totales train' : f"{len(tr):,}",
        'Vida media (ciclos)' : round(vida.mean(), 1),
        'Vida mínima'         : int(vida.min()),
        'Vida máxima'         : int(vida.max()),
        'Desv. estándar vida' : round(vida.std(), 1),
    })

resumen = pd.DataFrame(filas).set_index('Dataset')
print(resumen.to_string())

## 2. Calidad de datos y sensores constantes

El primer filtro analítico es verificar:

1. **Valores faltantes**: si los hay, hay que decidir entre imputación y eliminación.
2. **Sensores constantes**: un sensor cuya varianza es prácticamente cero a lo largo de toda la flota y toda su vida no aporta información discriminativa. Incluirlo incrementa la dimensionalidad sin beneficio y puede desestabilizar métodos basados en distancias (PCA, Mahalanobis).

El umbral de 0.01 sobre el desvío estándar global es conservador — un sensor con `std = 0.009` en una escala de rpm no aporta nada.

In [ ]:
# Valores faltantes
nulos = train_all[SENSORES].isnull().sum()
print('Valores nulos por sensor:')
print(nulos[nulos > 0].to_string() if nulos.any() else '  Ninguno.')

# Varianza global por sensor sobre el conjunto de entrenamiento completo
std_global = train_all[SENSORES].std().sort_values()

UMBRAL = 0.01
sensores_constantes   = std_global[std_global <  UMBRAL].index.tolist()
sensores_informativos = std_global[std_global >= UMBRAL].index.tolist()

print(f'\nSensores constantes (std < {UMBRAL}): {sensores_constantes}')
print(f'Sensores informativos ({len(sensores_informativos)}): {sensores_informativos}')

# Gráfica
fig = px.bar(
    x=std_global.index,
    y=std_global.values,
    color=(std_global < UMBRAL),
    color_discrete_map={True: '#d62728', False: '#1f77b4'},
    labels={'x': 'Sensor', 'y': 'Desvío estándar (todos los FDs, train)', 'color': 'Constante'},
    title='Varianza global por sensor — conjunto de entrenamiento',
    template='plotly_white',
)
fig.add_hline(y=UMBRAL, line_dash='dash', line_color='gray',
              annotation_text=f'Umbral = {UMBRAL}')
fig.show()

## 3. Condiciones de operación

Las tres columnas `os1` (altitud, kft), `os2` (Mach) y `os3` (temperatura estranguladora, °R) definen el punto de operación del motor en cada ciclo.

En **FD001 y FD003** existe una sola condición (vuelo a nivel del mar): los tres valores son constantes en toda la flota. En **FD002 y FD004** hay seis condiciones que reflejan distintas fases de vuelo: despegue, crucero, aproximación, etc.

Este hecho es crítico para el modelado. Si se mezclan condiciones sin normalizar, los sensores muestran distribuciones multimodales — cada moda corresponde a un régimen distinto — y el modelo aprende a distinguir regímenes en lugar de degradación. La solución estándar es asignar un régimen operacional a cada ciclo (por KMeans o por tabla de referencia) y normalizar los sensores dentro de cada régimen.

In [ ]:
# Muestra estratificada para no saturar la gráfica 3D
muestra_os = (
    train_all
    .groupby('dataset', group_keys=False)
    .apply(lambda g: g.sample(min(len(g), 1500), random_state=42))
)

fig = px.scatter_3d(
    muestra_os,
    x='os1', y='os2', z='os3',
    color='dataset',
    opacity=0.35,
    labels={'os1': 'Altitud (kft)', 'os2': 'Mach', 'os3': 'Temp. (°R)'},
    title='Condiciones de operación — todos los subconjuntos',
    template='plotly_white',
    width=800, height=600,
)
fig.update_traces(marker_size=2)
fig.show()

In [ ]:
# KMeans sobre os1-os2-os3 para identificar los 6 regímenes en FD002
fd2 = datasets['FD002']['train'].copy()

km = KMeans(n_clusters=6, random_state=0, n_init='auto')
fd2['regimen'] = km.fit_predict(fd2[['os1', 'os2', 'os3']]).astype(str)

centros = pd.DataFrame(km.cluster_centers_, columns=['os1', 'os2', 'os3'])
centros.index.name = 'Régimen'
print('Centros de régimen operacional (FD002):')
print(centros.round(3).to_string())

# Anotar cuántos ciclos caen en cada régimen
cuentas = fd2['regimen'].value_counts().sort_index()
print(f'\nCiclos por régimen: {cuentas.to_dict()}')

fig = px.scatter_3d(
    fd2.sample(3000, random_state=42),
    x='os1', y='os2', z='os3',
    color='regimen',
    opacity=0.5,
    title='FD002 — 6 regímenes identificados por KMeans',
    labels={'os1': 'Altitud (kft)', 'os2': 'Mach', 'os3': 'Temp. (°R)', 'regimen': 'Régimen'},
    template='plotly_white',
    width=800, height=600,
)
fig.update_traces(marker_size=3)
fig.show()

## 4. Distribución de vida útil por motor

La distribución del número de ciclos operacionales hasta el fallo (vida útil) determina:

- **Tamaño máximo de la ventana deslizante**: debe ser menor que la vida del motor más corto para garantizar al menos una muestra por motor.
- **Grado de desbalance de clases**: si la vida mínima es de 128 ciclos y la etiqueta `distress` se activa en los últimos 20, todos los motores tienen ciclos en ambas clases, pero la proporción es ~10–15% distress.
- **Heterogeneidad de la flota**: alta dispersión en la vida útil indica que el inicio de la degradación varía significativamente entre motores, lo que hace la predicción más difícil.

In [ ]:
vida_por_motor = (
    train_all
    .groupby(['dataset', 'unit'])['cycle']
    .max()
    .reset_index()
    .rename(columns={'cycle': 'vida_util'})
)

fig = px.histogram(
    vida_por_motor,
    x='vida_util',
    facet_col='dataset',
    facet_col_wrap=2,
    nbins=35,
    title='Distribución de vida útil por motor — conjuntos de entrenamiento',
    labels={'vida_util': 'Ciclos de vida', 'count': 'Cantidad de motores'},
    template='plotly_white',
    width=900, height=560,
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.update_layout(showlegend=False)
fig.show()

print(vida_por_motor.groupby('dataset')['vida_util'].describe().round(1).to_string())

## 5. Degradación temporal — tendencia media de sensores

El principio fundamental del PHM (Prognostics and Health Management) es que los sensores registran el deterioro físico antes del fallo. Si esto ocurre, al graficar el valor medio de un sensor contra la **vida normalizada** del motor, se debe observar una tendencia monótona.

La vida normalizada se define como:

$$\tilde{t} = \frac{t - 1}{t_{\max} - 1} \in [0, 1]$$

donde $t = 1$ es el primer ciclo y $t = t_{\max}$ es el ciclo de fallo.

Sensores con tendencia visible son candidatos directos como características del modelo. Sensores sin tendencia (ruido puro) aportan poco. El análisis se hace sobre **FD001** (una sola condición operacional) para no confundir degradación con cambio de régimen.

In [ ]:
fd1 = datasets['FD001']['train'].copy()

# Vida normalizada por motor
vida_max = fd1.groupby('unit')['cycle'].transform('max')
fd1['vida_norm'] = (fd1['cycle'] - 1) / (vida_max - 1).clip(lower=1)

# Discretizar en 25 intervalos para suavizar el ruido
fd1['vida_bin'] = pd.cut(fd1['vida_norm'], bins=25, labels=False)

tendencia = fd1.groupby('vida_bin')[sensores_informativos].mean()

# Normalizar cada sensor a [0, 1] para comparar en una sola escala
tendencia_norm = (tendencia - tendencia.min()) / (tendencia.max() - tendencia.min() + 1e-9)

# Seleccionar sensores con mayor rango de variación a lo largo de la vida
rango = tendencia.max() - tendencia.min()
sensores_tendencia = rango.nlargest(8).index.tolist()

fig = go.Figure()
for sensor in sensores_tendencia:
    fig.add_trace(go.Scatter(
        x=tendencia_norm.index / 25,
        y=tendencia_norm[sensor],
        mode='lines',
        name=f'{sensor}: {DESC_SENSOR.get(sensor, sensor)}',
        line=dict(width=2),
    ))

fig.update_layout(
    title='FD001 — Tendencia media normalizada de sensores vs. vida normalizada',
    xaxis_title='Vida normalizada (0 = inicio de operación, 1 = fallo)',
    yaxis_title='Valor medio normalizado [0–1]',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.85)', font_size=11),
    width=960, height=520,
)
fig.show()

print('Sensores con mayor variación a lo largo de la vida (FD001):')
print(rango[sensores_tendencia].round(2).to_string())

## 6. Variabilidad entre motores individuales

La tendencia media oculta la heterogeneidad entre motores: algunos se degradan rápidamente, otros lentamente. Esta variabilidad es el reto central del problema — el modelo debe generalizar a motores que no vio durante el entrenamiento.

Graficar trayectorias individuales de `s9` (velocidad del núcleo) revela si los patrones son homogéneos o muy dispersos, y si hay motores atípicos (outliers) que podrían distorsionar el entrenamiento.

In [ ]:
# Trayectorias de los primeros 20 motores de FD001
motores_vis = sorted(fd1['unit'].unique())[:20]
fd1_vis = fd1[fd1['unit'].isin(motores_vis)].copy()

fig = px.line(
    fd1_vis,
    x='cycle', y='s9',
    color='unit',
    labels={'cycle': 'Ciclo operacional', 's9': 's9 — Nc (rpm)', 'unit': 'Motor'},
    title='FD001 — Trayectorias individuales de s9 (primeros 20 motores)',
    template='plotly_white',
    width=960, height=480,
)
fig.update_traces(opacity=0.55, line_width=1.2)
fig.update_layout(showlegend=True, legend_title='Motor')
fig.show()

In [ ]:
# Media móvil de s9 por motor para visualizar la tendencia sin ruido
ventana = 10
fd1_vis = fd1_vis.sort_values(['unit', 'cycle'])
fd1_vis['s9_suav'] = (
    fd1_vis.groupby('unit')['s9']
    .transform(lambda x: x.rolling(ventana, min_periods=1).mean())
)

fig = px.line(
    fd1_vis,
    x='cycle', y='s9_suav',
    color='unit',
    labels={'cycle': 'Ciclo', 's9_suav': f's9 — media móvil ({ventana} ciclos)', 'unit': 'Motor'},
    title=f'FD001 — Tendencia suavizada de s9 (ventana = {ventana} ciclos)',
    template='plotly_white',
    width=960, height=480,
)
fig.update_traces(opacity=0.6, line_width=1.5)
fig.show()

## 7. Correlación entre sensores

Sensores altamente correlacionados miden esencialmente la misma variable física o están físicamente acoplados en el motor. Incluir ambos en el modelo no aporta información nueva pero sí incrementa la dimensionalidad y el riesgo de sobreajuste.

En un turbofán, es esperable que temperatura y presión de salida del HPC (`s3`, `s7`) estén correlacionadas con la velocidad del núcleo (`s9`), ya que todas reflejan el estado del mismo componente.

El mapa de calor de correlación de Pearson permite identificar grupos de sensores redundantes. Esto orienta la decisión de reducir dimensionalidad vía PCA o por selección manual.

In [ ]:
corr = fd1[sensores_informativos].corr()

fig = px.imshow(
    corr,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='FD001 — Correlación de Pearson entre sensores informativos',
    template='plotly_white',
    width=720, height=680,
    aspect='equal',
)
fig.update_layout(
    coloraxis_colorbar_title='r',
    font_size=10,
)
fig.show()

# Pares con correlación |r| > 0.9
corr_abs = corr.abs()
np.fill_diagonal(corr_abs.values, 0)
alto = [(i, j, round(corr.loc[i, j], 3))
        for i in corr_abs.index for j in corr_abs.columns
        if corr_abs.loc[i, j] > 0.9 and i < j]
print('Pares con |r| > 0.9 (altamente redundantes):')
for a, b, r in sorted(alto, key=lambda x: -abs(x[2])):
    print(f'  {a} — {b}: r = {r}')

## 8. Efecto de la condición operacional sobre los sensores (FD002)

Con seis condiciones operacionales, cada sensor muestra una distribución **multimodal**: cada moda corresponde a un régimen de vuelo distinto, no a un estado de salud diferente.

Esto hace que el mismo valor de `s9` pueda corresponder tanto a un motor sano en régimen de crucero como a un motor degradado en despegue. Cualquier modelo que ignore la condición operacional confunde estos dos estados.

La solución es normalizar los sensores **dentro de cada régimen**: restar la media del régimen y dividir por su desvío estándar. Esto convierte la pregunta de "¿cuál es el valor del sensor?" en "¿cuánto se desvía de lo esperado en este régimen?", que es la señal de degradación.

In [ ]:
# Distribución de sensores por régimen en FD002 (usando los regímenes de KMeans)
sensores_vis = [s for s in ['s2', 's9', 's11', 's14'] if s in sensores_informativos]

fig = make_subplots(
    rows=1, cols=len(sensores_vis),
    subplot_titles=[DESC_SENSOR.get(s, s) for s in sensores_vis],
)

colores = px.colors.qualitative.Plotly

for col_idx, sensor in enumerate(sensores_vis, start=1):
    for reg_idx, regimen in enumerate(sorted(fd2['regimen'].unique())):
        grupo = fd2[fd2['regimen'] == regimen][sensor]
        fig.add_trace(
            go.Box(
                y=grupo,
                name=f'R{regimen}',
                showlegend=(col_idx == 1),
                marker_color=colores[int(regimen)],
                boxpoints=False,
            ),
            row=1, col=col_idx,
        )

fig.update_layout(
    title='FD002 — Distribución de sensores por régimen operacional',
    template='plotly_white',
    width=1100, height=520,
    legend_title='Régimen',
)
fig.show()

## 9. Comparación entre modos de falla: FD001 vs. FD003

- **FD001**: un único modo de falla — degradación del compresor de alta presión (HPC)
- **FD003**: dos modos simultáneos — HPC + degradación del fan

La degradación del fan debería manifestarse principalmente en sensores del fan: `s8` (velocidad del fan, Nf), `s13` (Nf corregida), `s15` (bypass ratio). La del HPC afecta principalmente `s3`, `s7`, `s9`, `s11`.

Comparar las tendencias de degradación entre FD001 y FD003 permite identificar qué sensores son diagnósticos de cada modo de falla — información relevante si se quisiera construir un clasificador de causa raíz.

In [ ]:
def tendencia_normalizada(df, sensor, n_bins=25):
    """Calcula la tendencia media normalizada de un sensor sobre vida normalizada."""
    vida_max_col = df.groupby('unit')['cycle'].transform('max')
    vida_norm = (df['cycle'] - 1) / (vida_max_col - 1).clip(lower=1)
    vida_bin  = pd.cut(vida_norm, bins=n_bins, labels=False)
    serie     = df.groupby(vida_bin)[sensor].mean()
    serie_norm = (serie - serie.min()) / (serie.max() - serie.min() + 1e-9)
    return serie_norm


fd1_tr = datasets['FD001']['train']
fd3_tr = datasets['FD003']['train']

# Sensores relevantes para comparar HPC vs. Fan
sensores_comp = ['s9', 's11', 's13', 's15']

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'{s}: {DESC_SENSOR.get(s,s)}' for s in sensores_comp],
)

for idx, sensor in enumerate(sensores_comp):
    row, col = divmod(idx, 2)
    row += 1; col += 1

    for nombre, df, color in [
        ('FD001 (HPC)',      fd1_tr, '#1f77b4'),
        ('FD003 (HPC+Fan)', fd3_tr, '#d62728'),
    ]:
        t = tendencia_normalizada(df, sensor)
        fig.add_trace(
            go.Scatter(
                x=t.index / 25,
                y=t.values,
                mode='lines',
                name=nombre,
                legendgroup=nombre,
                showlegend=(idx == 0),
                line=dict(color=color, width=2),
            ),
            row=row, col=col,
        )

fig.update_layout(
    title='Tendencia media normalizada — FD001 (HPC) vs. FD003 (HPC + Fan)',
    template='plotly_white',
    width=950, height=600,
    legend=dict(x=0.5, y=-0.1, orientation='h'),
)
fig.update_xaxes(title_text='Vida normalizada')
fig.update_yaxes(title_text='Valor norm. [0–1]')
fig.show()

## 10. RUL ground truth en el conjunto de prueba

El conjunto de prueba no incluye trayectorias hasta el fallo. Cada motor se observa hasta un ciclo arbitrario antes del fallo; el archivo `RUL_FDxxx.txt` provee el RUL verdadero en ese último ciclo.

La distribución del RUL final en los conjuntos de prueba es relevante porque:

1. Si todos los RUL finales son altos (motor visto muy antes del fallo), la tarea de predicción es más difícil — el modelo debe extrapolar lejos.
2. Si los RUL finales son bajos (motor visto justo antes del fallo), la señal de degradación debería ser visible y la predicción más fácil.
3. La variabilidad en el RUL final afecta la distribución de errores al evaluar modelos de regresión de RUL.

In [ ]:
rul_list = []
for nombre, d in datasets.items():
    for motor_idx, rul_val in enumerate(d['rul']['RUL_final'], start=1):
        rul_list.append({'Dataset': nombre, 'Motor': motor_idx, 'RUL_final': rul_val})

rul_df = pd.DataFrame(rul_list)

fig = px.histogram(
    rul_df,
    x='RUL_final',
    facet_col='Dataset',
    facet_col_wrap=2,
    nbins=30,
    title='Distribución del RUL verdadero en conjuntos de prueba',
    labels={'RUL_final': 'RUL en el último ciclo observado (ciclos)', 'count': 'Motores'},
    template='plotly_white',
    width=900, height=560,
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.update_layout(showlegend=False)
fig.show()

print(rul_df.groupby('Dataset')['RUL_final'].describe().round(1).to_string())

## 11. Balance de clases para clasificación binaria

La etiqueta `distress = (RUL < 20)` identifica el período crítico antes del fallo. Esta etiqueta está desbalanceada por construcción: un motor que opera 200 ciclos pasa 180 ciclos en estado normal y solo 20 en distress — una proporción de 9:1.

Este desbalance tiene consecuencias directas en el entrenamiento:

- Un clasificador trivial que siempre predice "normal" alcanza ~88–90% de exactitud sin aprender nada útil.
- La exactitud (*accuracy*) es una métrica engañosa para este problema.
- Métricas apropiadas: F1-score, Precision-Recall AUC, balanced accuracy.
- Técnicas de mitigación: pesos de clase inversamente proporcionales a la frecuencia, oversampling SMOTE, umbral de clasificación ajustado.

In [ ]:
balance = (
    train_all
    .groupby('dataset')['distress']
    .value_counts(normalize=True)
    .rename('proporcion')
    .reset_index()
)
balance['clase'] = balance['distress'].map({0: 'Normal (RUL ≥ 20)', 1: 'Distress (RUL < 20)'})

fig = px.bar(
    balance,
    x='dataset', y='proporcion',
    color='clase',
    color_discrete_map={
        'Normal (RUL ≥ 20)'    : '#1f77b4',
        'Distress (RUL < 20)'  : '#d62728',
    },
    title='Proporción de clases por subconjunto — entrenamiento',
    labels={'proporcion': 'Proporción', 'dataset': 'Subconjunto', 'clase': 'Clase'},
    template='plotly_white',
    barmode='stack',
    text_auto='.1%',
)
fig.update_layout(width=700, height=460, legend_title='Clase')
fig.show()

# Tabla de conteos absolutos
conteos = (
    train_all
    .groupby(['dataset', 'distress'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: 'Normal', 1: 'Distress'})
)
conteos['Total'] = conteos.sum(axis=1)
conteos['% Distress'] = (conteos['Distress'] / conteos['Total'] * 100).round(1)
print(conteos.to_string())

## 12. Estacionariedad de las series de sensores

La prueba de Dickey-Fuller Aumentada (ADF) contrasta la hipótesis nula de que la serie temporal tiene raíz unitaria (no estacionaria, con tendencia):

- **p-valor < 0.05**: se rechaza la hipótesis nula → la serie es estacionaria en torno a una media constante.
- **p-valor ≥ 0.05**: no se puede rechazar → la serie tiene tendencia o raíz unitaria.

Para sensores con tendencia de degradación monótona, es esperable que la serie **no** sea estacionaria: el valor cambia sistemáticamente a lo largo de la vida. Esto justifica:

1. Usar diferenciación (parámetro `d=1` en ARIMA) para modelar cambios en lugar de niveles absolutos.
2. Modelar los residuales respecto a un ajuste lineal en lugar de la serie cruda.
3. Usar detección de anomalías basada en desviación respecto a una ventana de referencia (sana).

In [ ]:
# ADF sobre sensor_9 en los primeros 15 motores de FD001
resultados = []
for motor in sorted(fd1['unit'].unique())[:15]:
    serie = fd1[fd1['unit'] == motor]['s9'].values
    if len(serie) > 20:
        stat, pval, _, _, _, _ = adfuller(serie, maxlag=5, autolag='AIC')
        resultados.append({
            'Motor'            : int(motor),
            'Longitud (ciclos)': int(len(serie)),
            'Estadístico ADF'  : round(stat, 4),
            'p-valor'          : round(pval, 4),
            'Estacionaria'     : 'Sí' if pval < 0.05 else 'No',
        })

adf_df = pd.DataFrame(resultados).set_index('Motor')
print('Prueba ADF sobre s9 — FD001:')
print(adf_df.to_string())

fig = px.strip(
    adf_df.reset_index(),
    x='p-valor', y='Motor',
    color='Estacionaria',
    color_discrete_map={'Sí': '#2ca02c', 'No': '#d62728'},
    title='FD001 — p-valores ADF para s9 por motor (n=15)',
    template='plotly_white',
    width=700, height=430,
)
fig.add_vline(x=0.05, line_dash='dash', line_color='gray',
              annotation_text='α = 0.05', annotation_position='top right')
fig.update_layout(xaxis_range=[0, max(0.1, adf_df['p-valor'].max() * 1.1)])
fig.show()

## 13. Varianza por sensor: ranking definitivo

Una visualización comparativa de la varianza de cada sensor por subconjunto permite verificar si los sensores informativos son los mismos en FD001–FD004, o si la presencia de múltiples condiciones operacionales (FD002, FD004) infla artificialmente la varianza de ciertos sensores.

In [ ]:
# Desvío estándar por sensor y por dataset
std_por_dataset = (
    train_all
    .groupby('dataset')[SENSORES]
    .std()
    .T
    .reset_index()
    .rename(columns={'index': 'sensor'})
    .melt(id_vars='sensor', var_name='dataset', value_name='std')
)

# Ordenar por std promedio a través de datasets
orden_sensores = (
    std_por_dataset
    .groupby('sensor')['std']
    .mean()
    .sort_values(ascending=True)
    .index.tolist()
)

fig = px.bar(
    std_por_dataset,
    x='std', y='sensor',
    color='dataset',
    orientation='h',
    barmode='group',
    category_orders={'sensor': orden_sensores},
    title='Desvío estándar por sensor y subconjunto — entrenamiento',
    labels={'std': 'Desvío estándar', 'sensor': 'Sensor', 'dataset': 'Subconjunto'},
    template='plotly_white',
    width=900, height=700,
)
fig.show()

## Síntesis y decisiones de diseño para el modelado

### Hallazgos clave del análisis

| Aspecto | Hallazgo | Implicación |
|---------|----------|-------------|
| **Sensores constantes** | s1, s5, s10, s16, s18, s19 tienen varianza ≈ 0 | Eliminar antes de cualquier modelo |
| **Condiciones operacionales** | FD002 y FD004 tienen 6 regímenes distintos | Normalizar por régimen (KMeans sobre os1–os3) |
| **Vida útil** | Media ~200 ciclos; mínimo ~128 (FD001) | Ventana deslizante segura: ≤ 30 ciclos |
| **Degradación visible** | s2, s3, s4, s9, s11, s14, s17 muestran tendencia monótona | Candidatos principales como características |
| **Correlación** | Grupos s2-s3-s4, s9-s14, s11-s12 son altamente redundantes | PCA o selección manual reduce a ~6–8 características independientes |
| **Balance de clases** | ~10–12% de registros en clase distress | Usar F1, PR-AUC; ajustar pesos de clase; no evaluar con exactitud |
| **Estacionariedad** | s9 no estacionaria en la mayoría de los motores | ARIMA con d=1; o modelar residuales respecto a ventana de referencia |
| **Modos de falla** | FD001 vs. FD003 muestran firmas distintas en s13, s15 | Un clasificador multi-modo requeriría datos de ambos subconjuntos |

---

### Estrategia de preprocesamiento recomendada

1. **Eliminar** sensores constantes: `s1, s5, s10, s16, s18, s19`
2. **Para FD002/FD004**: asignar régimen con KMeans(k=6) sobre `os1–os3`; ajustar `StandardScaler` por régimen en datos de entrenamiento solamente
3. **Para FD001/FD003**: ajustar `StandardScaler` global en datos de entrenamiento
4. **Ventana deslizante**: 30 ciclos por motor, desplazamiento de 1 ciclo
5. **Validación cruzada**: `TimeSeriesSplit` sobre IDs de motor (orden cronológico), nunca mezclar motores entre train y validation

---

### Secuencia de modelado sugerida

| Orden | Modelo | Propósito |
|-------|--------|----------|
| 1 | Regresión logística sobre características estadísticas de ventana | Baseline interpretable |
| 2 | Random Forest | Captura no linealidades, da importancia de características |
| 3 | ARIMA + detección por residuales | Detección de anomalía sin supervisión |
| 4 | LSTM | Modela dependencia temporal directamente sobre la secuencia |
| 5 | LSTM + Attention | Permite identificar qué ciclos son más informativos |